Newton의 Actuator/Viewer 기능을 활용해서 기존의 - [단일 모터 디지털 트윈](code/single_motor_twin.py)보다 조금 더 구조화 한다. 

## Import
현재는 단일 무부하 모터에 대해 디지털 트윈을 구성하기에, franka의 예시처럼 `newton.examples`, `newton.utils`, `newton.ik` 과 같은 기능은 필요하지 않다.

- `ControllerPID`  
서보 드라이브의 제어기 역할. `Command Position - Feedback Position` 오차를 보고 kp, ki, kd를 이용해서 얼마만큼의 토크를 낼지 계산하는 함수
- `ClampingMaxEffort`  
PID가 계산한 토크가 실제 모터의 최대 토크를 넘지 못하게 자르는 함수
- `SolverMuJoCo`  
위에서 계산된 토크를 받은 1-DOF 모터 축이 실제로 어떻게 움직이는지 물리 계산하려고 넣은 함수


In [4]:
from dataclasses import dataclass

import numpy as np
import warp as wp
import newton

wp.config.log_level = wp.LOG_WARNING

from newton.actuators import ControllerPID, ClampingMaxEffort
from newton.solvers import SolverMuJoCo

## Motor Parameter Setup

**MotorParams**는 단일 모터 디지털 트윈에서 사용할 제어 및 물리 파라미터를 하나로 묶어 관리하기 위한 class이다.   
현재 설정된 초기값은 실제 모터의 확정값이 아닌 추후 **WMX3 실측 데이터와 비교하여 조정할 피팅 대상 값**이다. 

제어기 관련 파라미터 :
- `kp`, `ki`, `kd`
- `integral_max`
- `effort_limit`
- `delay_steps`

모터 물리 특성 관련 파라미터 :
- `inertia`
- `viscous`
- `coulomb`

In [ ]:
@dataclass
class MotorParams:
    # ---- 피팅 대상 파라미터 (초기값은 자리표시자) ----
    kp: float = 8.0            # 위치 (오차에 비례해 토크를 생성하는) 비례 게인
    ki: float = 0.0            # (누적된 위치 오차를 보정하는) 적분 게인
    kd: float = 0.6            # (모터의 속도에 따른 감쇠 역할을 하는) 미분(속도) 게인
    integral_max: float = 5.0  # 적분값이 과도하게 누적되는 것을 제한하는 anti-windup 한계
    effort_limit: float = 5.0  # 토크 포화 [N·m] : 모터가 출력 가능한 토크의 최대값
    inertia: float = 0.02      # 유효(반사) 관성 J [kg·m^2]
    viscous: float = 0.01      # 점성 마찰 b [N·m·s/rad] : 속도에 비례해 발생
    coulomb: float = 0.0       # 쿨롱 마찰 [N·m] : 운동 방향의 반대 방향으로 작용
    delay_steps: int = 1       # Command Position이 실제 제어 계산에 반영되기까지의 지연 사이클

## Build a Single Motor Model

articulation은 로보틱스/물리 시뮬레이션에서 여러 rigid body(강체)가 joint(관절)로 연결된 하나의 관절 시스템을 뜻함.  

회전축 하나만 있는 articulation으로 만들기.  
=> 고정된 베이스와 회전하는 body를 하나의 revolute joint로 연결해서 1-DOF 관절 시스템을 만든다.
